# 01 — Exploratory Data Analysis
## Bank Customer Churn — European Dataset

This notebook performs a comprehensive EDA of the European Bank Customer dataset.
Each chart is followed by a written insight explaining what it shows and why it matters for retention strategy.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from preprocessing import load_data, TARGET
from feature_engineering import add_engineered_features

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'sans-serif',
    'font.size': 11,
})
PALETTE = ['#10B981', '#EF4444']  # Green=retained, Red=churned

FIGURES_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

df = load_data()
df = add_engineered_features(df)
print(f'Dataset shape: {df.shape}')
print(f'Churn rate: {df[TARGET].mean():.2%}')
df.head()

## 1. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df[TARGET].value_counts()
axes[0].bar(['Retained', 'Churned'], counts.values, color=PALETTE, edgecolor='white', linewidth=1.5)
for i, (label, val) in enumerate(zip(['Retained', 'Churned'], counts.values)):
    axes[0].text(i, val + 30, f'{val:,}\n({val/len(df):.1%})', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Customer Count by Churn Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Customers')

# Pie chart
axes[1].pie(counts.values, labels=['Retained\n(79.6%)', 'Churned\n(20.4%)'],
            colors=PALETTE, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Customer Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: The dataset exhibits moderate class imbalance with approximately 20.4% churn rate.')
print('This requires class-weighting or oversampling (SMOTE) during model training to prevent')
print('the classifier from simply predicting the majority class. Standard accuracy metrics')
print('are misleading here; ROC-AUC is the primary evaluation metric.')

## 2. Churn Rate by Geography

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

geo_churn = df.groupby('Geography')[TARGET].agg(['mean', 'count']).reset_index()
geo_churn.columns = ['Geography', 'Churn_Rate', 'Count']

colors_geo = ['#3B82F6', '#EF4444', '#10B981']
axes[0].bar(geo_churn['Geography'], geo_churn['Churn_Rate'] * 100, color=colors_geo, edgecolor='white')
for i, (geo, rate) in enumerate(zip(geo_churn['Geography'], geo_churn['Churn_Rate'])):
    axes[0].text(i, rate * 100 + 0.5, f'{rate:.1%}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Geography', fontsize=13, fontweight='bold')
axes[0].axhline(df[TARGET].mean() * 100, color='black', linestyle='--', label='Overall avg')
axes[0].legend()

# Stacked bar
geo_pivot = df.groupby(['Geography', TARGET]).size().unstack()
geo_pivot.plot(kind='bar', stacked=True, ax=axes[1], color=PALETTE[::-1],
               edgecolor='white', linewidth=0.5)
axes[1].set_title('Customer Count by Geography and Churn', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Geography')
axes[1].legend(['Retained', 'Churned'], loc='upper right')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_geography_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Germany stands out with a ~32% churn rate — approximately 2x the rate of France (~16%)')
print('and Spain (~17%). This geographic disparity is one of the most striking findings in the dataset.')
print('German customers may face different competitive offers, service quality issues, or market dynamics.')
print('Germany should receive disproportionate retention budget allocation.')

## 3. Churn Rate by Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gen_churn = df.groupby('Gender')[TARGET].mean() * 100
axes[0].bar(gen_churn.index, gen_churn.values, color=['#EC4899', '#3B82F6'], edgecolor='white', linewidth=1.5)
for i, (gen, rate) in enumerate(zip(gen_churn.index, gen_churn.values)):
    axes[0].text(i, rate + 0.3, f'{rate:.1f}%', ha='center', fontsize=13, fontweight='bold')
axes[0].axhline(df[TARGET].mean() * 100, color='black', linestyle='--', label='Overall avg')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Gender', fontsize=13, fontweight='bold')
axes[0].legend()

# Cross-tab
ct = pd.crosstab(df['Gender'], df[TARGET], normalize='index') * 100
ct.plot(kind='bar', stacked=False, ax=axes[1], color=PALETTE[::-1], edgecolor='white')
axes[1].set_title('Churn vs Retention % by Gender', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Retained', 'Churned'])
axes[1].set_ylabel('%')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_gender_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Female customers churn at ~25%, compared to ~16% for male customers — a 9-percentage-point gap.')
print('This difference warrants careful consideration: gender should not be used as a sole decision variable')
print('in interventions (regulatory constraint), but the underlying behavioural differences should be')
print('investigated (e.g., product relevance, service interaction preferences, financial life stages).')

## 4. Churn by Number of Products

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

prod_stats = df.groupby('NumOfProducts').agg(
    Churn_Rate=(TARGET, 'mean'),
    Count=(TARGET, 'count')
).reset_index()

bars = ax.bar(prod_stats['NumOfProducts'], prod_stats['Churn_Rate'] * 100,
               color=['#10B981', '#3B82F6', '#F59E0B', '#EF4444'], edgecolor='white', linewidth=1.5)
ax2 = ax.twinx()
ax2.plot(prod_stats['NumOfProducts'], prod_stats['Count'], 'k--o', markersize=8, linewidth=2, label='Count')
ax2.set_ylabel('Number of Customers', fontsize=11)

for bar, rate, count in zip(bars, prod_stats['Churn_Rate'], prod_stats['Count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate:.0%}', ha='center', fontsize=13, fontweight='bold')

ax.set_xlabel('Number of Products', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_title('Churn Rate by Number of Products (Non-linear!)', fontsize=13, fontweight='bold')
ax.axhline(df[TARGET].mean() * 100, color='grey', linestyle=':', label='Overall avg')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_products_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: NumOfProducts shows a striking non-linear U-shape:')
print('  1 product:  ~28% churn — high risk, clear cross-sell opportunity')
print('  2 products: ~8%  churn — sweet spot, strongest retention')
print('  3 products: ~83% churn — paradoxical spike, possible mis-selling or fee fatigue')
print('  4 products: ~100% churn — nearly all churned (very small segment)')
print('Key action: Focus cross-sell on 1-product customers (add exactly 1 relevant product).')
print('Investigate why 3+ product customers churn at such extreme rates.')

## 5. Churn by Active Member Status and Credit Card

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, labels, title in zip(
    axes,
    ['IsActiveMember', 'HasCrCard'],
    [['Inactive', 'Active'], ['No Credit Card', 'Has Credit Card']],
    ['Churn Rate by Active Member Status', 'Churn Rate by Credit Card Ownership'],
):
    churn_by_col = df.groupby(col)[TARGET].mean() * 100
    bars = ax.bar(labels, churn_by_col.values, color=['#EF4444', '#10B981'], edgecolor='white', linewidth=1.5)
    for bar, rate in zip(bars, churn_by_col.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{rate:.1f}%', ha='center', fontsize=14, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axhline(df[TARGET].mean() * 100, color='black', linestyle='--', alpha=0.5, label='Overall avg')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_engagement_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: IsActiveMember is one of the strongest binary churn predictors.')
print('  Inactive members: ~27% churn rate')
print('  Active members:   ~14% churn rate (nearly half the risk)')
print('This is the most directly actionable lever: re-engagement programmes can directly')
print('move customers from inactive to active status, halving their churn probability.')
print()
print('HasCrCard shows minimal predictive power — churn rates are similar regardless of')
print('credit card ownership. This feature may contribute little to models.')

## 6. Age Distribution vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by churn
for exited, label, color in [(0, 'Retained', '#10B981'), (1, 'Churned', '#EF4444')]:
    subset = df[df[TARGET] == exited]['Age']
    axes[0].hist(subset, bins=30, alpha=0.65, color=color, label=label, density=True)
axes[0].set_xlabel('Age', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Age Distribution by Churn Status', fontsize=13, fontweight='bold')
axes[0].legend()

# Churn rate by age band
df['Age_Band_'] = pd.cut(df['Age'], bins=[0,30,40,50,60,120], labels=['18-30','31-40','41-50','51-60','60+'])
age_churn = df.groupby('Age_Band_', observed=True)[TARGET].mean() * 100
axes[1].bar(age_churn.index, age_churn.values,
             color=['#10B981', '#3B82F6', '#F59E0B', '#EF4444', '#8B5CF6'],
             edgecolor='white', linewidth=1.5)
for i, (band, rate) in enumerate(zip(age_churn.index, age_churn.values)):
    axes[1].text(i, rate + 0.5, f'{rate:.1f}%', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Age Band', fontsize=13, fontweight='bold')
axes[1].axhline(df[TARGET].mean() * 100, color='black', linestyle='--', label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_age_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Age shows a clear non-monotonic relationship with churn.')
print('  Young customers (18-30): relatively low churn — they are still establishing relationships')
print('  Middle-aged (41-60):     peak churn — life changes, greater financial awareness, more options')
print('  Older customers (60+):   moderate churn — established, but may simplify their financial lives')
print('Age-specific retention campaigns are warranted, particularly for the 41-60 cohort.')

## 7. Balance and Salary Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

numeric_vars = ['Balance', 'EstimatedSalary', 'CreditScore', 'Tenure']
for ax, var in zip(axes.flatten(), numeric_vars):
    for exited, label, color in [(0, 'Retained', '#10B981'), (1, 'Churned', '#EF4444')]:
        subset = df[df[TARGET] == exited][var]
        ax.hist(subset, bins=40, alpha=0.60, color=color, label=label, density=True)
    ax.set_xlabel(var, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{var} Distribution by Churn Status', fontsize=12, fontweight='bold')
    ax.legend()

plt.suptitle('Numeric Feature Distributions by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_numeric_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT:')
print('  Balance: Churned customers have a bimodal distribution — a spike at zero AND a spread across')
print('           mid-to-high balances. Retained customers are more concentrated at lower balances.')
print('  Salary:  Similar distributions for churned and retained — salary alone is not very predictive.')
print('  CreditScore: Slight left-shift for churned customers (lower credit scores on average).')
print('  Tenure:  Near-uniform distribution for both groups — tenure alone does not strongly predict churn.')

## 8. Boxplots: Numeric Features vs Churn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

numeric_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

for ax, col in zip(axes.flatten(), numeric_cols):
    df.boxplot(column=col, by=TARGET, ax=ax, patch_artist=True,
               medianprops={'color': 'black', 'linewidth': 2},
               boxprops={'facecolor': '#BFDBFE'},
               flierprops={'marker': 'o', 'markerfacecolor': '#94A3B8', 'markersize': 3, 'alpha': 0.5})
    ax.set_title(f'{col} by Churn', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(['Retained (0)', 'Churned (1)'])
    plt.sca(ax)
    plt.title(f'{col} by Churn', fontsize=11, fontweight='bold')

plt.suptitle('', fontsize=1)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Boxplots reveal:')
print('  Age:          Churned customers are notably older (median ~45 vs ~37 for retained)')
print('  Balance:      Higher IQR for churned — more spread, including many zeros')
print('  CreditScore:  Slightly lower for churned customers')
print('  Tenure:       Similar medians — tenure alone is not strongly predictive')
print('  Salary:       Nearly identical distributions — salary has low predictive power')

## 9. Correlation Heatmap

In [ ]:
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
                    'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited',
                    'Balance_to_Salary', 'Products_per_Tenure', 'Active_Products',
                    'Age_Tenure_Interaction', 'Tenure_Age_Ratio', 'Zero_Balance_Flag']
numeric_features = [c for c in numeric_features if c in df.columns]

corr = df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Correlation Heatmap — All Features (including Engineered)',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Key correlations with Exited (target):')
print('  Age (r~0.29):             Strongest linear positive correlation with churn')
print('  IsActiveMember (r~-0.16): Negative correlation — active members less likely to churn')
print('  Balance (r~0.12):         Moderate positive correlation')
print('  Zero_Balance_Flag:        Engineered feature with meaningful correlation to churn')
print('  Active_Products (r~-0.18): Interaction feature captures combined engagement+product effect')
print()
print('Low inter-feature correlations confirm that most features provide independent information.')
print('No severe multicollinearity issues are present (no r > 0.7 between predictors).')

## 10. Tenure Churn Rate

In [ ]:
tenure_churn = df.groupby('Tenure')[TARGET].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(tenure_churn.index, tenure_churn.values, color='#3B82F6', edgecolor='white', linewidth=1.2)
ax.plot(tenure_churn.index, tenure_churn.values, 'o-', color='#1D4ED8', linewidth=2.5, markersize=7)
ax.axhline(df[TARGET].mean() * 100, color='red', linestyle='--', label='Overall avg')
ax.set_xlabel('Tenure (Years)', fontsize=12)
ax.set_ylabel('Churn Rate (%)', fontsize=12)
ax.set_title('Churn Rate by Customer Tenure', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_tenure_churn.png'), dpi=150, bbox_inches='tight')
plt.show()
print()
print('INSIGHT: Tenure shows a relatively flat churn rate across years (0–10),')
print('indicating that length of relationship alone is not strongly predictive.')
print('However, tenure interacted with age (Tenure_Age_Ratio) provides stronger signal,')
print('as older customers with short tenures are at higher risk than young customers with short tenures.')

## 11. EDA Summary

| Finding | Retention Implication |
|---|---|
| Germany: 32% churn rate | Increase retention budget for German market |
| Female: 25% vs Male: 16% | Investigate product-gender fit |
| 1-product customers: 28% churn | Cross-sell exactly 1 relevant product |
| Inactive members: 27% churn | Re-engagement campaign priority |
| Age 45-60: ~30% churn peak | Life-stage financial planning offers |
| Zero balance: elevated risk | Incentivised minimum balance programme |
| Salary: low discriminatory power | Deprioritise salary in interventions |